# Face Mask Detection via Image Classification

PyTorch + timm + Albumentations

## 2. Project Overview

This project builds a face mask classifier on a cropped-face dataset and explains when classification is the right framing versus when object detection is better.

## 3. Learning Objectives

- Build a mask classifier with transfer learning
- Use timm backbones and Albumentations augmentation
- Evaluate with confusion matrix and class-wise metrics
- Compare classification and detection design choices

## 4. Problem Statement

Given a cropped face image, predict whether the person is wearing a mask or not.

## 5. Why This Project Matters

Mask-status recognition can support safety monitoring workflows, but design choice depends heavily on whether faces are already cropped or appear in full scenes.

## 6. Dataset Overview

This project uses a face-mask dataset organized into class folders such as `with_mask` and `without_mask`, which strongly suggests a classification setup rather than a detection-first pipeline.

## 7. Dataset Source and License Notes

Use the original dataset only according to its license terms and privacy constraints. Face datasets require extra care because they contain biometric imagery.

## 8. Environment Setup

Required: torch, timm, albumentations, scikit-learn, matplotlib, seaborn.
Optional alternative framing: Ultralytics/RT-DETR when starting from uncropped multi-person scene images.

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

warnings.filterwarnings('ignore')

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('timm:', timm.__version__)
print('Albumentations:', A.__version__)

In [ ]:
# 10. Configuration / constants
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 160
BATCH_SIZE = 32
NUM_WORKERS = 0
LR = 1e-4
EPOCHS_BASELINE = 1
EPOCHS_STRONG = 1

BASELINE_MODEL = 'resnet18'
STRONG_MODEL = 'efficientnet_b0'

SAVE_DIR = Path.cwd() / 'Computer Vision' / 'Face Mask Detection'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR = SAVE_DIR / 'Source Code'

print('Device:', DEVICE)
print('Save dir:', SAVE_DIR)
print('Source dir exists:', SOURCE_DIR.exists())

In [ ]:
# 11. Dataset loading
FORCE_SYNTHETIC = False
use_synthetic = FORCE_SYNTHETIC

class_names = ['without_mask', 'with_mask']
class_to_idx = {name: i for i, name in enumerate(class_names)}

train_images, train_labels = [], []
val_images, val_labels = [], []
test_images, test_labels = [], []

train_root = SOURCE_DIR / 'faces'
valid_root = SOURCE_DIR / 'valid'


def load_images_from_folder(folder_path, label, limit=None):
    items = []
    if not folder_path.exists():
        return items
    files = sorted(folder_path.iterdir())
    if limit is not None:
        files = files[:limit]
    for p in files:
        if p.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}:
            continue
        img = cv2.imread(str(p))
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        items.append((img, label))
    return items

if train_root.exists() and valid_root.exists() and not use_synthetic:
    # Use actual dataset with small caps so notebook executes quickly.
    train_items = []
    train_items.extend(load_images_from_folder(train_root / 'without_mask', class_to_idx['without_mask'], limit=240))
    train_items.extend(load_images_from_folder(train_root / 'with_mask', class_to_idx['with_mask'], limit=240))

    valid_items = []
    valid_items.extend(load_images_from_folder(valid_root / 'without_mask', class_to_idx['without_mask'], limit=120))
    valid_items.extend(load_images_from_folder(valid_root / 'with_mask', class_to_idx['with_mask'], limit=120))

    np.random.shuffle(train_items)
    np.random.shuffle(valid_items)

    split = len(valid_items) // 2
    val_split = valid_items[:split]
    test_split = valid_items[split:]

    for img, y in train_items:
        train_images.append(img)
        train_labels.append(y)
    for img, y in val_split:
        val_images.append(img)
        val_labels.append(y)
    for img, y in test_split:
        test_images.append(img)
        test_labels.append(y)
else:
    use_synthetic = True

if use_synthetic:
    n_train = 480
    n_val = 120
    n_test = 120
    train_labels = np.random.choice([0, 1], size=n_train, p=[0.45, 0.55]).tolist()
    val_labels = np.random.choice([0, 1], size=n_val, p=[0.45, 0.55]).tolist()
    test_labels = np.random.choice([0, 1], size=n_test, p=[0.45, 0.55]).tolist()

    for _ in range(n_train):
        train_images.append(np.random.randint(0, 255, size=(IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
    for _ in range(n_val):
        val_images.append(np.random.randint(0, 255, size=(IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
    for _ in range(n_test):
        test_images.append(np.random.randint(0, 255, size=(IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))

print('Using synthetic mode:', use_synthetic)
print('Train/Val/Test sizes:', len(train_labels), len(val_labels), len(test_labels))

In [ ]:
# 12. Data validation checks
assert len(train_images) == len(train_labels)
assert len(val_images) == len(val_labels)
assert len(test_images) == len(test_labels)
assert set(train_labels).issubset({0, 1})

print('Validation checks passed.')
print('Train class counts:', dict(zip(class_names, np.bincount(np.array(train_labels), minlength=2).tolist())))

## 13. Exploratory Data Analysis

Inspect class counts and sample cropped faces.

In [ ]:
counts = np.bincount(np.array(train_labels), minlength=2)

plt.figure(figsize=(5,4))
plt.bar(class_names, counts)
plt.title('Train Class Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(10, 6))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(train_images[i])
    ax.set_title(class_names[train_labels[i]])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 14. Train/Validation/Test Split Strategy

When valid folders already exist, use them for validation and carve out test only from held-out data. Avoid mixing the same captured subject across splits when metadata is available.

In [ ]:
# 15. Preprocessing and augmentation strategy
train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.08, rotate_limit=12, border_mode=cv2.BORDER_REFLECT_101, p=0.4),
    A.RandomBrightnessContrast(0.18, 0.18, p=0.3),
    A.GaussNoise(var_limit=(4, 20), p=0.15),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

print('Albumentations pipelines configured.')

## 16. Baseline Approach

Baseline classifier: ResNet-18 transfer learning.

In [ ]:
class MaskDataset(Dataset):
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx]
        y = int(self.labels[idx])
        x = self.transform(image=img)['image']
        return x, y

train_ds = MaskDataset(train_images, train_labels, train_tf)
val_ds = MaskDataset(val_images, val_labels, val_tf)
test_ds = MaskDataset(test_images, test_labels, val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

baseline_model = timm.create_model(BASELINE_MODEL, pretrained=True, num_classes=2).to(DEVICE)
strong_model = timm.create_model(STRONG_MODEL, pretrained=True, num_classes=2).to(DEVICE)

print('Baseline model:', BASELINE_MODEL)
print('Strong model:', STRONG_MODEL)
print('Data loaders ready.')

## 17. Main Model/Workflow

Stronger classifier: EfficientNet-B0 transfer learning.

In [ ]:
# 18. Training loop / fine-tuning pipeline
def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    ys, ps = [], []

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        if train_mode:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train_mode):
            logits = model(xb)
            loss = criterion(logits, yb)
            if train_mode:
                loss.backward()
                optimizer.step()

        total_loss += float(loss.item())
        pred = logits.argmax(dim=1)
        ys.extend(yb.cpu().numpy().tolist())
        ps.extend(pred.cpu().numpy().tolist())

    avg_loss = total_loss / max(len(loader), 1)
    acc = accuracy_score(ys, ps)
    f1 = f1_score(ys, ps, zero_division=0)
    rec = recall_score(ys, ps, zero_division=0)
    return avg_loss, acc, f1, rec, ys, ps


def train_model(model, name, epochs):
    optimizer = optim.AdamW(model.parameters(), lr=LR)
    best_f1 = -1.0
    best_state = None

    for ep in range(1, epochs + 1):
        tr_loss, tr_acc, tr_f1, tr_rec, _, _ = run_epoch(model, train_loader, optimizer=optimizer)
        va_loss, va_acc, va_f1, va_rec, _, _ = run_epoch(model, val_loader, optimizer=None)
        print(f'[{name}] ep={ep} train_acc={tr_acc:.4f} val_acc={va_acc:.4f} val_f1={va_f1:.4f}')
        if va_f1 > best_f1:
            best_f1 = va_f1
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

train_model(baseline_model, 'baseline', EPOCHS_BASELINE)
train_model(strong_model, 'strong', EPOCHS_STRONG)

## 19. Inference Examples

Deployment-style classification response on one face crop.

In [ ]:
def infer_single(model, image_rgb):
    model.eval()
    x = val_tf(image=image_rgb)['image'].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    pred = int(np.argmax(probs))
    return {
        'predicted_label': pred,
        'predicted_name': class_names[pred],
        'confidence': float(probs[pred]),
        'class_probabilities': {class_names[i]: float(probs[i]) for i in range(len(class_names))}
    }

sample = test_images[0]
resp = infer_single(strong_model, sample)
print(json.dumps(resp, indent=2))

## 20. Evaluation

Report accuracy, precision, recall, F1, and confusion matrix.

In [ ]:
_, b_acc, b_f1, b_rec, by, bp = run_epoch(baseline_model, test_loader, optimizer=None)
_, s_acc, s_f1, s_rec, sy, sp = run_epoch(strong_model, test_loader, optimizer=None)

b_prec = precision_score(by, bp, zero_division=0)
s_prec = precision_score(sy, sp, zero_division=0)

print('Baseline -> acc:', round(b_acc,4), 'precision:', round(b_prec,4), 'recall:', round(b_rec,4), 'f1:', round(b_f1,4))
print('Strong   -> acc:', round(s_acc,4), 'precision:', round(s_prec,4), 'recall:', round(s_rec,4), 'f1:', round(s_f1,4))

print()
print('Classification report (strong):')
print(classification_report(sy, sp, target_names=class_names, zero_division=0))

## 21. Confusion Matrix and Error Analysis

In [ ]:
cm = confusion_matrix(sy, sp, labels=[0,1])
cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

plt.figure(figsize=(5,4))
sns.heatmap(cmn, annot=True, fmt='.3f', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Normalized Confusion Matrix (Strong Model)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

print('False positives (without_mask -> with_mask):', int(cm[0,1]))
print('False negatives (with_mask -> without_mask):', int(cm[1,0]))

## 22. Task Framing Decision

**Why classification is the better framing here:**

This dataset is already organized as cropped face images in `with_mask` and `without_mask` folders. That means the localization problem is already solved upstream. A classifier only needs to decide the label of one face crop.

Use **classification** when:
1. each image contains one cropped face
2. the training labels are image-level class labels
3. deployment pipeline already includes face detection/tracking

Use **detection** when:
1. images are full scenes with multiple people
2. you need both face localization and mask status
3. you need one-pass inference on surveillance or crowd images

## 23. Classification vs Detection Design Comparison

**Classification design**
- Input: cropped face
- Output: one label (`with_mask` / `without_mask`)
- Model choices: timm classifiers
- Pros: simpler, lighter, easier to train, needs fewer annotations
- Cons: requires a separate face detector upstream

**Detection design**
- Input: full image or video frame
- Output: bounding boxes + mask labels
- Model choices: Ultralytics YOLO / RT-DETR
- Pros: solves localization and classification jointly
- Cons: needs box annotations, heavier training/inference, more complex evaluation

For this dataset, classification is the correct primary framing. Detection becomes the better framing only if the task shifts to uncropped real-world scenes.

## 24. How To Improve

- Add a third class such as incorrectly worn mask if dataset supports it
- Calibrate probabilities for alerting thresholds
- Chain with a face detector for real-time scene deployment

## 25. Production Considerations

- Respect privacy and biometric-data handling requirements
- Monitor false negatives if the system is used for safety enforcement
- If moving to video scenes, switch to a detector instead of stretching a classifier beyond its design

## 26. Common Mistakes

- Using a classifier directly on full scene images
- Ignoring dataset bias in face angle, lighting, and demographics
- Confusing detection annotations with classification labels

## 27. Mini Challenge / Final Summary

Mini challenge: add a face-detection front end and compare pipeline latency against a single-stage detector.

Summary: this notebook builds the right solution for a cropped-face mask dataset and explicitly explains when to move to detection.

In [ ]:
# Save metrics
metrics = {
    'dataset': 'Face Mask Detection dataset',
    'task_framing': 'classification',
    'baseline_model': BASELINE_MODEL,
    'strong_model': STRONG_MODEL,
    'baseline_test_accuracy': float(b_acc),
    'baseline_test_precision': float(b_prec),
    'baseline_test_recall': float(b_rec),
    'baseline_test_f1': float(b_f1),
    'strong_test_accuracy': float(s_acc),
    'strong_test_precision': float(s_prec),
    'strong_test_recall': float(s_rec),
    'strong_test_f1': float(s_f1),
    'false_positives_strong': int(cm[0,1]),
    'false_negatives_strong': int(cm[1,0]),
    'device': str(DEVICE),
    'synthetic_mode': bool(use_synthetic)
}

metrics_path = SAVE_DIR / 'metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Saved metrics:', metrics_path)
print('Done.')